In [ ]:
import numpy as np

# For reproducibility of our "random" acoustic features
np.random.seed(42)

print("--- Phase 1, Step 1: Defining the 'Ground Truth' ---")

# 1. Define the True Latent Sequence (the "script")
z_true = ["lei", "sure", "or", "time"]
print(f"True syllable sequence: {z_true}\n")


# 2. Define True Neutral-Rate Durations (gamma_i) in seconds
gamma_true = {
    "lei": 0.20,
    "sure": 0.20,
    "or": 0.08,  # The function word is naturally shorter
    "time": 0.25
}
print(f"True neutral-rate durations (gamma): {gamma_true}\n")


# 3. Define True Emission Parameters (the "sound" of each syllable)
# According to your mentor's suggestion, our feature vector is 25-dimensional.
n_features = 25

# True mean acoustic vector for each syllable (mu_i)
# We'll create distinct random vectors for each syllable's "sound".
mu_true = {s: np.random.rand(n_features) for s in z_true}

# True covariance matrix for each syllable (sigma_i^2)
# For simplicity, we'll assume the features are independent (diagonal covariance)
# and have the same small variance.
variance = 0.1
sigma_true = {s: np.identity(n_features) * variance for s in z_true}

print(f"Acoustic feature dimensions: {n_features}")
print(f"Example mean vector for 'or' (mu_or): {mu_true['or'][:5]}...") # Print first 5 features
print(f"Covariance matrix for 'or' is a {sigma_true['or'].shape} identity matrix.\n")


# 4. Define True Base Transitions (pi_bar_i)
# Based on the mentor's idea to get this from the "pros" or script.
# The transitions are deterministic in this simple case.
pi_bar_true = {
    "lei":  {"sure": 1.0},
    "sure": {"or": 1.0},
    "or":   {"time": 1.0},
    "time": {} # No state follows "time"
}
print(f"True base transition probabilities (pi_bar): {pi_bar_true}\n")

In [ ]:
# We are continuing from the previous block.
# The 'true' parameter variables are already in memory.

print("\n--- Phase 1, Step 2: Generating Acoustic Datasets ---")

def generate_acoustic_data(z_sequence, gamma_params, mu_params, sigma_params, rate, time_step_duration=0.01):
    """
    Generates a continuous time series of acoustic data based on a sequence of states.

    Args:
        z_sequence (list): The sequence of true latent states (syllables).
        gamma_params (dict): Dictionary of neutral-rate durations for each state.
        mu_params (dict): Dictionary of mean acoustic vectors for each state.
        sigma_params (dict): Dictionary of covariance matrices for each state.
        rate (float): The speech rate in syllables per second.
        time_step_duration (float): The duration of a single time step in seconds.

    Returns:
        tuple: A tuple containing:
            - np.ndarray: The generated acoustic time series (n_steps, n_features).
            - list: The true state label for each time step.
    """
    acoustic_series = []
    state_label_series = []

    print(f"\nGenerating data for rate = {rate} syllables/sec...")

    for syllable in z_sequence:
        # Get the true parameters for the current syllable
        gamma = gamma_params[syllable]
        mu = mu_params[syllable]
        sigma = sigma_params[syllable]

        # Calculate the duration and number of time steps for this syllable instance
        duration_in_seconds = gamma / rate
        num_time_steps = max(1, round(duration_in_seconds / time_step_duration)) # Ensure at least 1 time step
        print(f"  - Syllable '{syllable}': duration = {duration_in_seconds:.3f}s, time steps = {num_time_steps}")

        # Generate acoustic data for this syllable's duration
        for _ in range(num_time_steps):
            # Draw one sample from the syllable's acoustic distribution
            acoustic_sample = np.random.multivariate_normal(mu, sigma)
            acoustic_series.append(acoustic_sample)
            state_label_series.append(syllable)

    return np.array(acoustic_series), state_label_series

# --- Set the two different speech rates ---
r_slow = 4.0  # syllables/sec
r_fast = 7.0  # syllables/sec

# --- Generate the two datasets ---
a_slow, z_slow_true_labels = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_slow)
a_fast, z_fast_true_labels = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_fast)

print("\n--- Data Generation Complete ---")
print(f"Slow dataset shape (a_slow): {a_slow.shape}")
print(f"Fast dataset shape (a_fast): {a_fast.shape}")
print(f"Example true labels for the slow data: {z_slow_true_labels[18:28]}")
print(f"Example true labels for the fast data: {z_fast_true_labels[10:20]}")

In [ ]:
import numpy as np
from scipy.stats import gamma as gamma_dist, norm, binom

# (Continuing with the same class structure)

class RateHMM:
    # __init__ and other methods remain the same as the previous step
    def __init__(self, states, n_features, gamma_hyperparams, mu0_hyperparams, sigma_hyperparams, pi_bar_hyperparams):
        """Initializes the Rate-Dependent Sticky HMM."""
        self.states = states
        self.state_map = {name: i for i, name in enumerate(states)}
        self.idx_to_state = {i: name for name, i in self.state_map.items()}
        self.n_states = len(states)
        self.n_features = n_features
        self.gamma_hyperparams = gamma_hyperparams
        self.mu0_hyperparams = mu0_hyperparams
        self.sigma_hyperparams = sigma_hyperparams
        self.pi_bar_hyperparams = pi_bar_hyperparams
        self.gamma = {s: gamma_dist.rvs(a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta']) for s in self.states}
        self.mu = {s: np.random.normal(loc=self.mu0_hyperparams['mu0'], scale=self.mu0_hyperparams['tau0'], size=n_features) for s in self.states}
        self.sigma_sq = {s: 1.0 / gamma_dist.rvs(a=self.sigma_hyperparams['a'], scale=1/self.sigma_hyperparams['b']) for s in self.states}
        self.pi_bar = {}
        for s_from_name in self.states:
            alpha_vec = self.pi_bar_hyperparams[s_from_name]
            self.pi_bar[s_from_name] = np.random.dirichlet(alpha_vec)

    # _get_transition_matrix, _get_emission_log_probs, _sample_latent_states,
    # _sample_emissions, and _sample_transitions remain the same.

    def _sample_gamma(self, z_sequence, r, proposal_std=0.01):
        """Samples gamma for each state using a Metropolis-Hastings step."""
        for state in self.states:
            # 1. Get current gamma and count transitions for this state
            current_gamma = self.gamma[state]

            n_self_transitions = 0
            n_total_from_state = 0
            for t in range(len(z_sequence) - 1):
                if z_sequence[t] == state:
                    n_total_from_state += 1
                    if z_sequence[t+1] == state:
                        n_self_transitions += 1

            if n_total_from_state == 0:
                continue # Cannot update if state was not observed

            # 2. Propose a new gamma from a small normal distribution around the current one
            proposed_gamma = np.random.normal(loc=current_gamma, scale=proposal_std)
            if proposed_gamma <= 0:
                continue # gamma must be positive, reject proposal

            # 3. Calculate acceptance probability (in log space for stability)
            # Log-Likelihood of data given parameters
            # The likelihood of n_self_transitions out of n_total_from_state is a Binomial distribution
            # with probability p = kappa = gamma/r.
            current_kappa = np.clip(current_gamma / r, 0, 1)
            proposed_kappa = np.clip(proposed_gamma / r, 0, 1)

            log_lik_current = binom.logpmf(n_self_transitions, n_total_from_state, current_kappa)
            log_lik_proposed = binom.logpmf(n_self_transitions, n_total_from_state, proposed_kappa)

            # Log-Prior probability of parameters
            log_prior_current = gamma_dist.logpdf(current_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])
            log_prior_proposed = gamma_dist.logpdf(proposed_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])

            # Log-Acceptance Ratio
            log_acceptance_ratio = (log_lik_proposed + log_prior_proposed) - (log_lik_current + log_prior_current)

            # 4. Decide to accept or reject
            if np.log(np.random.rand()) < log_acceptance_ratio:
                self.gamma[state] = proposed_gamma # Accept the proposal


    def fit(self, a_data, r, num_iterations=100):
        # The fit loop is now complete and will call the real _sample_gamma
        posterior_samples = []
        print(f"\n--- Starting MCMC Fitting for rate r={r} ---")
        for i in range(num_iterations):
            if i % 10 == 0: print(f"  Iteration {i}/{num_iterations}...")
            # 1. Sample latent states
            z_sequence = self._sample_latent_states(a_data, r)
            # 2. Sample parameters
            self._sample_emissions(a_data, z_sequence)
            self._sample_transitions(z_sequence)
            self._sample_gamma(z_sequence, r) # Now fully implemented
            # Store results
            current_params = {'gamma': self.gamma.copy(), 'mu': self.mu.copy(), 'sigma_sq': self.sigma_sq.copy(), 'pi_bar': self.pi_bar.copy(), 'z': z_sequence}
            posterior_samples.append(current_params)
        print("--- MCMC Fitting Complete ---")
        return posterior_samples

In [ ]:
from collections import Counter

# We are continuing from the previous block.
# The complete RateHMM class and the datasets a_slow and a_fast are in memory.

print("\n--- Phase 3: Running Full Simulation and Analyzing Results ---")

# --- 1. Fit the model to the SLOW dataset ---
hmm_slow = RateHMM(
    states=z_true,
    n_features=n_features,
    gamma_hyperparams=hyperparams['gamma'],
    mu0_hyperparams=hyperparams['mu0'],
    sigma_hyperparams=hyperparams['sigma'],
    pi_bar_hyperparams=pi_bar_init_params
)
# We'll use more iterations for a more stable result
posterior_samples_slow = hmm_slow.fit(a_slow, r_slow, num_iterations=200)


# --- 2. Fit the model to the FAST dataset ---
hmm_fast = RateHMM(
    states=z_true,
    n_features=n_features,
    gamma_hyperparams=hyperparams['gamma'],
    mu0_hyperparams=hyperparams['mu0'],
    sigma_hyperparams=hyperparams['sigma'],
    pi_bar_hyperparams=pi_bar_init_params
)
posterior_samples_fast = hmm_fast.fit(a_fast, r_fast, num_iterations=200)


# --- 3. Analyze the Inferred State Sequences ---
# For each simulation, we find the single most probable state sequence from the posterior samples.
# We can do this by taking the mode of the sampled 'z' sequence at each time step
# across the last half of the iterations (to ensure the sampler has converged).

def get_most_probable_sequence(posterior_samples):
    # Discard the first half of samples as "burn-in"
    burn_in = len(posterior_samples) // 2
    z_samples = [s['z'] for s in posterior_samples[burn_in:]]

    # Find the most common state at each time step
    num_time_steps = len(z_samples[0])
    most_probable_z = []
    for t in range(num_time_steps):
        time_step_states = [z[t] for z in z_samples]
        most_common_state = Counter(time_step_states).most_common(1)[0][0]
        most_probable_z.append(most_common_state)

    # To make it more readable, we'll return the sequence of unique states
    unique_sequence = []
    for state in most_probable_z:
        if not unique_sequence or unique_sequence[-1] != state:
            unique_sequence.append(state)

    return unique_sequence

inferred_sequence_slow = get_most_probable_sequence(posterior_samples_slow)
inferred_sequence_fast = get_most_probable_sequence(posterior_samples_fast)


# --- 4. Final Validation ---
print("\n\n--- FINAL RESULTS ---")
print(f"Ground Truth Sequence: {z_true}")
print("-" * 25)
print(f"Inferred Sequence (Fast Rate): {inferred_sequence_fast}")
print(f"Inferred Sequence (Slow Rate): {inferred_sequence_slow}")
print("-" * 25)

# Check our hypothesis
if "or" in inferred_sequence_fast and "or" not in inferred_sequence_slow:
    print("\n✅ Success! The model correctly perceived the function word 'or' in the fast context,")
    print("   but missed it in the slow context, explaining the 'disappearing word' phenomenon.")
else:
    print("\n❌ The simulation did not produce the expected result. Further tuning of hyperparameters or the model may be needed.")

# Combined

In [ ]:
import numpy as np
from scipy.stats import gamma as gamma_dist, norm, dirichlet, binom
from collections import Counter

# For reproducibility
np.random.seed(42)

print("--- Phase 1, Step 1: Defining the 'Ground Truth' ---")

# 1. Define the True Latent Sequence (the "script")
z_true = ["lei", "sure", "or", "time"]
print(f"True syllable sequence: {z_true}\n")

# 2. Define True Neutral-Rate Durations (gamma_i) in seconds
gamma_true = {
    "lei": 0.20,
    "sure": 0.20,
    "or": 0.08,  # The function word is naturally shorter
    "time": 0.25
}
print(f"True neutral-rate durations (gamma): {gamma_true}\n")

# 3. Define True Emission Parameters (the "sound" of each syllable)
n_features = 25
mu_true = {s: np.random.rand(n_features) for s in z_true}
variance = 0.1
sigma_true = {s: np.identity(n_features) * variance for s in z_true}
print(f"Acoustic feature dimensions: {n_features}")
print(f"Example mean vector for 'or' (mu_or): {mu_true['or'][:5]}...\n")

# 4. Define True Base Transitions (pi_bar_i)
pi_bar_true = {
    "lei":  {"sure": 1.0},
    "sure": {"or": 1.0},
    "or":   {"time": 1.0},
    "time": {}
}
print(f"True base transition probabilities (pi_bar): {pi_bar_true}\n")


print("\n--- Phase 1, Step 2: Generating Acoustic Datasets ---")

def generate_acoustic_data(z_sequence, gamma_params, mu_params, sigma_params, rate, time_step_duration=0.01):
    acoustic_series = []
    state_label_series = []
    print(f"\nGenerating data for rate = {rate} syllables/sec...")
    for syllable in z_sequence:
        gamma_val = gamma_params[syllable]
        mu = mu_params[syllable]
        sigma = sigma_params[syllable]
        duration_in_seconds = gamma_val / rate
        num_time_steps = max(1, round(duration_in_seconds / time_step_duration))
        print(f"  - Syllable '{syllable}': duration = {duration_in_seconds:.3f}s, time steps = {num_time_steps}")
        for _ in range(num_time_steps):
            acoustic_sample = np.random.multivariate_normal(mu, sigma)
            acoustic_series.append(acoustic_sample)
            state_label_series.append(syllable)
    return np.array(acoustic_series), state_label_series

r_slow = 4.0
r_fast = 7.0
a_slow, z_slow_true_labels = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_slow)
a_fast, z_fast_true_labels = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_fast)

print("\n--- Data Generation Complete ---")
print(f"Slow dataset shape (a_slow): {a_slow.shape}")
print(f"Fast dataset shape (a_fast): {a_fast.shape}\n")


print("\n--- Phase 2: Defining the Model and Hyperparameters ---")

# Define the hyperparameters for the model's priors
hyperparams = {
    'gamma': {'alpha': 2.0, 'beta': 10.0}, # Vaguely centered around 0.2
    'mu0': {'mu0': 0.5, 'tau0': 0.5}, # Centered at 0.5
    'sigma': {'a': 1.0, 'b': 10.0}, # Corresponds to variance of 0.1
}

# Define the dirichlet hyperparameters for the base transitions
# Small values indicate uncertainty about the next state
pi_bar_init_params = {
    "lei":  (1, 1, 0.1, 0.1),
    "sure": (0.1, 1, 1, 0.1),
    "or":   (0.1, 0.1, 1, 1),
    "time": (0.1, 0.1, 0.1, 1), # High self-transition prob for terminal state
}

class RateHMM:
    def __init__(self, states, n_features, gamma_hyperparams, mu0_hyperparams, sigma_hyperparams, pi_bar_hyperparams):
        """Initializes the Rate-Dependent Sticky HMM."""
        self.states = states
        self.state_map = {name: i for i, name in enumerate(states)}
        self.idx_to_state = {i: name for name, i in self.state_map.items()}
        self.n_states = len(states)
        self.n_features = n_features

        # Store hyperparameters
        self.gamma_hyperparams = gamma_hyperparams
        self.mu0_hyperparams = mu0_hyperparams
        self.sigma_hyperparams = sigma_hyperparams
        self.pi_bar_hyperparams = pi_bar_hyperparams

        # Initialize parameters from priors
        self.gamma = {s: gamma_dist.rvs(a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta']) for s in self.states}
        self.mu = {s: np.random.normal(loc=self.mu0_hyperparams['mu0'], scale=self.mu0_hyperparams['tau0'], size=n_features) for s in self.states}
        self.sigma_sq = {s: 1.0 / gamma_dist.rvs(a=self.sigma_hyperparams['a'], scale=1/self.sigma_hyperparams['b']) for s in self.states}
        self.pi_bar = {s_from: dirichlet.rvs(alpha=self.pi_bar_hyperparams[s_from])[0] for s_from in self.states}

    def _get_transition_matrix(self, r):
        T = np.zeros((self.n_states, self.n_states))
        for i, s_from in enumerate(self.states):
            kappa = np.clip(self.gamma[s_from] / r, 0, 1)
            base_transitions = self.pi_bar[s_from]
            T[i, :] = (1 - kappa) * base_transitions
            T[i, i] += kappa
        return T

    def _get_emission_log_probs(self, a_data):
        log_probs = np.zeros((len(a_data), self.n_states))
        for t, a_t in enumerate(a_data):
            for i, state in enumerate(self.states):
                log_probs[t, i] = norm.logpdf(a_t, loc=self.mu[state], scale=np.sqrt(self.sigma_sq[state])).sum()
        return log_probs

    def _sample_latent_states(self, a_data, r):
        # Forward-Backward sampling
        T_matrix = self._get_transition_matrix(r)
        log_emissions = self._get_emission_log_probs(a_data)

        # Forward pass
        alpha = np.zeros((len(a_data), self.n_states))
        alpha[0, :] = np.log(np.ones(self.n_states) / self.n_states) + log_emissions[0, :]
        for t in range(1, len(a_data)):
            for j in range(self.n_states):
                log_sum_exp = np.logaddexp.reduce(alpha[t-1, :] + np.log(T_matrix[:, j]))
                alpha[t, j] = log_sum_exp + log_emissions[t, j]

        # Backward sampling
        z_sequence_indices = np.zeros(len(a_data), dtype=int)
        p = np.exp(alpha[-1, :] - np.logaddexp.reduce(alpha[-1, :]))
        z_sequence_indices[-1] = np.random.choice(self.n_states, p=p)
        for t in range(len(a_data) - 2, -1, -1):
            prev_z = z_sequence_indices[t+1]
            log_p = alpha[t, :] + np.log(T_matrix[:, prev_z])
            p = np.exp(log_p - np.logaddexp.reduce(log_p))
            z_sequence_indices[t] = np.random.choice(self.n_states, p=p)

        return [self.idx_to_state[i] for i in z_sequence_indices]

    def _sample_emissions(self, a_data, z_sequence):
        tau0_sq = self.mu0_hyperparams['tau0']**2
        mu0 = self.mu0_hyperparams['mu0']
        a_sig = self.sigma_hyperparams['a']
        b_sig = self.sigma_hyperparams['b']

        for state in self.states:
            state_indices = [t for t, s in enumerate(z_sequence) if s == state]
            n_k = len(state_indices)
            if n_k == 0: continue

            a_k = a_data[state_indices]
            a_k_bar = a_k.mean(axis=0)

            # Sample mu
            var_post = 1 / (1/tau0_sq + n_k/self.sigma_sq[state])
            mean_post = var_post * (mu0/tau0_sq + (n_k*a_k_bar)/self.sigma_sq[state])
            self.mu[state] = np.random.normal(loc=mean_post, scale=np.sqrt(var_post))

            # Sample sigma_sq
            a_post = a_sig + (n_k * self.n_features) / 2
            b_post = b_sig + 0.5 * np.sum((a_k - self.mu[state])**2)
            self.sigma_sq[state] = 1.0 / gamma_dist.rvs(a=a_post, scale=1/b_post)

    def _sample_transitions(self, z_sequence):
        for s_from in self.states:
            s_from_idx = self.state_map[s_from]
            counts = np.zeros(self.n_states)
            for t in range(len(z_sequence) - 1):
                if z_sequence[t] == s_from and z_sequence[t+1] != s_from:
                    s_to_idx = self.state_map[z_sequence[t+1]]
                    counts[s_to_idx] += 1
            alpha_post = self.pi_bar_hyperparams[s_from] + counts
            self.pi_bar[s_from] = dirichlet.rvs(alpha=alpha_post)[0]

    def _sample_gamma(self, z_sequence, r, proposal_std=0.01):
        """Samples gamma for each state using a Metropolis-Hastings step."""
        for state in self.states:
            current_gamma = self.gamma[state]
            n_self_transitions = 0
            n_total_from_state = 0
            for t in range(len(z_sequence) - 1):
                if z_sequence[t] == state:
                    n_total_from_state += 1
                    if z_sequence[t+1] == state:
                        n_self_transitions += 1
            if n_total_from_state == 0: continue

            proposed_gamma = np.random.normal(loc=current_gamma, scale=proposal_std)
            if proposed_gamma <= 0: continue

            current_kappa = np.clip(current_gamma / r, 1e-9, 1-1e-9)
            proposed_kappa = np.clip(proposed_gamma / r, 1e-9, 1-1e-9)

            log_lik_current = binom.logpmf(n_self_transitions, n_total_from_state, current_kappa)
            log_lik_proposed = binom.logpmf(n_self_transitions, n_total_from_state, proposed_kappa)

            log_prior_current = gamma_dist.logpdf(current_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])
            log_prior_proposed = gamma_dist.logpdf(proposed_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])

            log_acceptance_ratio = (log_lik_proposed + log_prior_proposed) - (log_lik_current + log_prior_current)
            if np.log(np.random.rand()) < log_acceptance_ratio:
                self.gamma[state] = proposed_gamma

    def fit(self, a_data, r, num_iterations=100):
        posterior_samples = []
        print(f"\n--- Starting MCMC Fitting for rate r={r} ---")
        for i in range(num_iterations):
            if i % 20 == 0: print(f"  Iteration {i}/{num_iterations}...")
            z_sequence = self._sample_latent_states(a_data, r)
            self._sample_emissions(a_data, z_sequence)
            self._sample_transitions(z_sequence)
            self._sample_gamma(z_sequence, r)
            if i > num_iterations // 2: # Burn-in
                posterior_samples.append({'z': z_sequence})
        print("--- MCMC Fitting Complete ---")
        return posterior_samples


print("\n--- Phase 3: Running Full Simulation and Analyzing Results ---")

# --- 1. Fit the model to the SLOW dataset ---
hmm_slow = RateHMM(
    states=z_true, n_features=n_features, gamma_hyperparams=hyperparams['gamma'],
    mu0_hyperparams=hyperparams['mu0'], sigma_hyperparams=hyperparams['sigma'],
    pi_bar_hyperparams=pi_bar_init_params
)
posterior_samples_slow = hmm_slow.fit(a_slow, r_slow, num_iterations=200)

# --- 2. Fit the model to the FAST dataset ---
hmm_fast = RateHMM(
    states=z_true, n_features=n_features, gamma_hyperparams=hyperparams['gamma'],
    mu0_hyperparams=hyperparams['mu0'], sigma_hyperparams=hyperparams['sigma'],
    pi_bar_hyperparams=pi_bar_init_params
)
posterior_samples_fast = hmm_fast.fit(a_fast, r_fast, num_iterations=200)

# --- 3. Analyze the Inferred State Sequences ---
def get_most_probable_sequence(posterior_samples):
    if not posterior_samples: return ["No samples collected"]
    z_samples = [s['z'] for s in posterior_samples]
    num_time_steps = len(z_samples[0])
    most_probable_z = []
    for t in range(num_time_steps):
        time_step_states = [z[t] for z in z_samples]
        most_common_state = Counter(time_step_states).most_common(1)[0][0]
        most_probable_z.append(most_common_state)

    unique_sequence = []
    for state in most_probable_z:
        if not unique_sequence or unique_sequence[-1] != state:
            unique_sequence.append(state)
    return unique_sequence

inferred_sequence_slow = get_most_probable_sequence(posterior_samples_slow)
inferred_sequence_fast = get_most_probable_sequence(posterior_samples_fast)

# --- 4. Final Validation ---
print("\n\n--- FINAL RESULTS ---")
print(f"Ground Truth Sequence: {z_true}")
print("-" * 35)
print(f"Inferred Sequence (Fast Rate, r={r_fast}): {inferred_sequence_fast}")
print(f"Inferred Sequence (Slow Rate, r={r_slow}): {inferred_sequence_slow}")
print("-" * 35)
